# Lab 9: Premier Agent ADK pour Data Science

**Navigation** : [Lab 8 <<](../Day4-Foundations/Lab8-ADK-Introduction.ipynb) | [Index](../../README.md) | [>> Lab 10](../Day5-DS-Star/Lab10-File-Analyzer.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Créer un agent simple capable d'exécuter du code Python
2. Analyser un DataFrame pandas avec l'agent
3. Comparer avec l'approche LangChain (`create_pandas_dataframe_agent`)
4. Tester avec différents providers (vLLM, Gemini)

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec `ACTIVE_PROVIDER`
- Connaissance de base des agents (Lab 7 complété)

### Durée estimée : 40-50 minutes

## 1. Configuration de l'Environnement

Nous utilisons notre couche d'abstraction multi-provider pour créer un agent capable d'exécuter du code Python.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from io import StringIO
import json
import re

from config import get_settings, get_provider_config, ProviderType
from utils import LLMClient, generate

ModuleNotFoundError: No module named 'config'

Chargement de la configuration du provider.

In [2]:
# Chargement de la configuration
settings = get_settings()
config = get_provider_config(settings)

print(f"Provider: {config.provider.value}")
print(f"Modèle: {config.model}")

NameError: name 'get_settings' is not defined

## 2. Création d'un Dataset de Test

Nous créons un dataset de ventes simple pour tester notre agent.

In [3]:
# Création d'un dataset de ventes
np.random.seed(42)

data = {
    'date': pd.date_range('2024-01-01', periods=100, freq='D'),
    'product': np.random.choice(['Widget A', 'Widget B', 'Gadget X', 'Gadget Y'], 100),
    'region': np.random.choice(['Nord', 'Sud', 'Est', 'Ouest'], 100),
    'quantity': np.random.randint(1, 50, 100),
    'price': np.random.uniform(10, 100, 100).round(2)
}

df = pd.DataFrame(data)
df['revenue'] = df['quantity'] * df['price']

# Sauvegarde pour l'agent
df.to_csv('sales_data.csv', index=False)

print(f"Dataset créé: {len(df)} lignes")
df.head()

Dataset créé: 100 lignes


,date,product,region,quantity,price,revenue
0,2024-01-01,Gadget X,Est,32,11.49,367.68
1,2024-01-02,Gadget Y,Sud,39,56.09,2187.51
2,2024-01-03,Widget A,Sud,49,30.38,1488.62
3,2024-01-04,Gadget X,Ouest,32,68.07,2178.24
4,2024-01-05,Gadget X,Sud,4,25.69,102.76


## 3. Implémentation d'un Agent Simple avec Code Execution

Contrairement à LangChain qui fournit `create_pandas_dataframe_agent`, nous allons construire notre propre agent avec une capacité d'exécution de code Python.

### Architecture

```
┌─────────────┐
│   Prompt    │  Question utilisateur + contexte DataFrame
└──────┬──────┘
       │
       v
┌─────────────┐
│    LLM      │  Génère du code Python
└──────┬──────┘
       │
       v
┌─────────────┐
│  Executor   │  Exécute le code de façon sécurisée
└──────┬──────┘
       │
       v
┌─────────────┐
│   Output    │  Résultat + explication
└─────────────┘
```

In [4]:
class SimpleDataAgent:
    """
    Agent simple pour l'analyse de données avec exécution de code Python.
    
    Cette implémentation simplifiée montre les concepts clés d'un agent
    de data science sans utiliser de framework complexe.
    """
    
    def __init__(self, df: pd.DataFrame, client: LLMClient):
        """
        Initialise l'agent avec un DataFrame et un client LLM.
        
        Args:
            df: DataFrame pandas à analyser
            client: Client LLM pour la génération
        """
        self.df = df
        self.client = client
        self.df_name = 'df'
        
    def _get_system_prompt(self) -> str:
        """Génère le prompt système avec le contexte du DataFrame."""
        df_info = f"""DataFrame: {self.df_name}
Colonnes: {list(self.df.columns)}
Types: {self.df.dtypes.to_dict()}
Shape: {self.df.shape}
Premières lignes:
{self.df.head(3).to_string()}
Statistiques descriptives:
{self.df.describe().to_string()}
"""
        
        return f"""Tu es un expert en analyse de données. Tu as accès à un DataFrame pandas.

{df_info}

INSTRUCTIONS:
1. Pour répondre aux questions, génère UNIQUEMENT du code Python valide
2. Le code doit être entouré de balises ```python ... ```
3. Utilise print() pour afficher les résultats
4. Le DataFrame est disponible dans la variable '{self.df_name}'
5. Après le code, fournis une brève explication en français

EXEMPLE:
Question: Quelle est la moyenne des ventes?
Réponse:
```python
print(df['revenue'].mean())
```
La moyenne des revenus est affichée ci-dessus.
"""

    def _extract_code(self, response: str) -> str:
        """Extrait le code Python de la réponse du LLM."""
        # Recherche du code entre balises ```python ... ```
        pattern = r'```python\s*(.*?)\s*```'
        matches = re.findall(pattern, response, re.DOTALL)
        
        if matches:
            return matches[0]
        
        # Fallback: cherche n'importe quel bloc de code
        pattern = r'```\s*(.*?)\s*```'
        matches = re.findall(pattern, response, re.DOTALL)
        return matches[0] if matches else ""

    def _execute_code(self, code: str) -> str:
        """
        Exécute le code Python de façon sécurisée.
        
        WARNING: Cette implémentation est simplifiée pour l'enseignement.
        Pour la production, utilisez un sandbox (Docker, RestrictedPython, etc.)
        """
        # Création d'un namespace limité
        namespace = {
            'df': self.df,
            'pd': pd,
            'np': np,
            'print': print,
        }
        
        # Capture de la sortie
        from io import StringIO
        import sys
        
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        
        try:
            exec(code, namespace)
            output = sys.stdout.getvalue()
        except Exception as e:
            output = f"Erreur: {str(e)}"
        finally:
            sys.stdout = old_stdout
            
        return output
    
    def analyze(self, question: str) -> str:
        """
        Analyse le DataFrame en répondant à une question.
        
        Args:
            question: Question sur les données
            
        Returns:
            Réponse avec code exécuté et explication
        """
        # Génération de la réponse
        response = self.client.generate(
            prompt=question,
            system=self._get_system_prompt(),
            temperature=0.1  # Basse température pour du code déterministe
        )
        
        # Extraction et exécution du code
        code = self._extract_code(response)
        
        if code:
            output = self._execute_code(code)
            return {
                'response': response,
                'code': code,
                'output': output
            }
        
        return {
            'response': response,
            'code': None,
            'output': None
        }

NameError: name 'LLMClient' is not defined

## 4. Test de l'Agent

In [5]:
# Création de l'agent
client = LLMClient()
agent = SimpleDataAgent(df, client)

print("Agent créé avec succès!")
print(f"Provider: {client.config.provider.value}")
print(f"Modèle: {client.config.model}")

NameError: name 'LLMClient' is not defined

### Question 1: Statistiques de base

In [6]:
result = agent.analyze("Quel est le revenu total par région?")

print("=== CODE GÉNÉRÉ ===")
print(result['code'])
print("\n=== RÉSULTAT ===")
print(result['output'])
print("\n=== RÉPONSE COMPLETE ===")
print(result['response'])

NameError: name 'agent' is not defined

### Question 2: Analyse temporelle

In [7]:
result = agent.analyze("Quel est le produit le plus vendu par mois? Montre le top 3 pour chaque mois.")

print("=== CODE GÉNÉRÉ ===")
print(result['code'])
print("\n=== RÉSULTAT ===")
print(result['output'])

NameError: name 'agent' is not defined

### Question 3: Visualisation

In [8]:
result = agent.analyze("Crée un graphique à barres montrant le revenu par produit. Utilise matplotlib.")

print("=== CODE GÉNÉRÉ ===")
print(result['code'])
print("\n=== RÉSULTAT ===")
print(result['output'])

NameError: name 'agent' is not defined

## 5. Comparaison avec LangChain

### Approche LangChain

```python
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4")
agent = create_pandas_dataframe_agent(llm, df, verbose=True)
agent.invoke("Quel est le revenu total par région?")
```

### Différences Clés

| Aspect | Notre Agent Simple | LangChain Agent |
|--------|-------------------|-----------------|
| **Dépendances** | Minimal (litellm, pandas) | langes chaines de dépendances |
| **Contrôle** | Full contrôle sur l'exécution | Abstraction, moins visible |
| **Sécurité** | À implémenter soi-même | Intégré (PythonAstREPLTool) |
| **Multi-provider** | Natif via notre config | Nécessite adapters |
| **Debugging** | Facile (code visible) | Plus complexe |

### Avantages de Notre Approche

1. **Léger**: Pas de dépendances lourdes
2. **Pédagogique**: On comprend chaque composant
3. **Flexible**: Facile d'ajouter des tools personnalisés
4. **Multi-provider**: Fonctionne avec n'importe quel LLM

## 6. Amélioration: Ajout de Tools Supplémentaires

Étendons notre agent avec des tools spécialisés.

In [9]:
class EnhancedDataAgent(SimpleDataAgent):
    """
    Agent étendu avec des tools supplémentaires.
    """
    
    def _get_system_prompt(self) -> str:
        base_prompt = super()._get_system_prompt()
        
        tools_info = """
TOOLS DISPONIBLES:
1. df_info() - Affiche les infos du DataFrame
2. df_describe() - Statistiques descriptives
3. df_columns() - Liste des colonnes avec types
4. plot_bar(x, y, title) - Crée un graphique à barres
5. plot_line(x, y, title) - Crée un graphique linéaire
6. plot_pie(values, labels, title) - Crée un graphique circulaire
"""
        return base_prompt + tools_info
    
    def _execute_code(self, code: str) -> str:
        """Exécution avec outils additionnels."""
        import matplotlib.pyplot as plt
        
        # Tools helpers
        def df_info():
            return self.df.info()
        
        def df_describe():
            return self.df.describe()
        
        def df_columns():
            return self.df.dtypes
        
        def plot_bar(x, y, title=""):
            plt.figure(figsize=(10, 6))
            self.df.groupby(x)[y].sum().plot(kind='bar')
            plt.title(title)
            plt.tight_layout()
            plt.show()
            return f"Graphique créé: {title}"
        
        def plot_line(x, y, title=""):
            plt.figure(figsize=(10, 6))
            self.df.plot(x=x, y=y, kind='line')
            plt.title(title)
            plt.tight_layout()
            plt.show()
            return f"Graphique créé: {title}"
        
        def plot_pie(values, labels, title=""):
            plt.figure(figsize=(8, 8))
            self.df.groupby(labels)[values].sum().plot(kind='pie', autopct='%1.1f%%')
            plt.title(title)
            plt.show()
            return f"Graphique créé: {title}"
        
        namespace = {
            'df': self.df,
            'pd': pd,
            'np': np,
            'plt': plt,
            'df_info': df_info,
            'df_describe': df_describe,
            'df_columns': df_columns,
            'plot_bar': plot_bar,
            'plot_line': plot_line,
            'plot_pie': plot_pie,
            'print': print,
        }
        
        from io import StringIO
        import sys
        
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        
        try:
            exec(code, namespace)
            output = sys.stdout.getvalue()
        except Exception as e:
            output = f"Erreur: {str(e)}"
        finally:
            sys.stdout = old_stdout
            
        return output

NameError: name 'SimpleDataAgent' is not defined

Test de l'agent etendu avec des requetes plus complexes.

In [10]:
# Test de l'agent étendu
enhanced_agent = EnhancedDataAgent(df, client)

result = enhanced_agent.analyze("Utilise plot_pie pour montrer la répartition du revenu par région.")

print("=== CODE GÉNÉRÉ ===")
print(result['code'])
print("\n=== RÉSULTAT ===")
print(result['output'])

NameError: name 'EnhancedDataAgent' is not defined

## 7. Résumé et Points Clés

### Ce que nous avons appris

1. **Architecture d'un Agent**: Composants LLM, Executor, Tools
2. **Code Execution**: Exécuter du code généré par le LLM
3. **Tools**: Ajouter des fonctions spécialisées
4. **Multi-provider**: Fonctionne avec n'importe quel LLM configuré

### Sécurité (IMPORTANT)

⚠️ L'exécution de code générée par un LLM présente des risques:

- **Pour le développement**: Notre approche simple suffit
- **Pour la production**: Utilisez un sandbox (Docker, gVisor, RestrictedPython)
- **Jamais**: N'exécutez pas de code non validé sur des données sensibles

### Prochaines étapes

- **Lab 10**: File Analyzer de DS-STAR
- **Lab 11**: Boucle Planner-Coder-Verifier
- **Lab 12**: DS-STAR complet

## 8. Exercice

1. Posez 3 questions supplémentaires à l'agent
2. Ajoutez un tool `plot_histogram(column, bins=10)`
3. Testez avec un autre provider (changez `ACTIVE_PROVIDER` dans `.env`)

In [11]:
# Espace pour vos exercices

# Question 1:
# result = agent.analyze("Votre question ici")
# print(result['output'])

# Question 2:

# Question 3: